# Don't Repeat Yourself For Your Pipeline (DRY FYP)

## Impor Librari Pandas, Numpy, OS, Matplotlib, Seaborn, Scikit Learn, Warnings, Collections, Joblib, dan Datetime

In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn as skl
import warnings
import collections
import joblib

from datetime import datetime as dttm

warnings.filterwarnings('ignore')

filepath = None

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filepath is None:
            filepath = os.path.join(dirname, filename)

## Pendefinisian Class `PreprocessingPipeline` Khusus Dataset Convenience Store

In [2]:
class PreprocessingPipeline:

    def get_dataframe(self, split, target, dropped_columns):
        if split:
            X = self.__preprocessing_dataframe.drop(columns=dropped_columns)
            y = self.__preprocessing_dataframe[target]
            
            tts = skl.model_selection.train_test_split
            
            return tts(X, y, test_size=0.2, shuffle=False)
        else:
            return self.dataframe        

    def __preprocess(self, X, y, role="train"):

        y = np.log10(y // 10 + 1)

        X_ = X.groupby("Customer_Name").size().reset_index()
        X_ = X_.rename(columns={0: 'Transaction_Counts'})
        
        def _loyalty_classes_(x):
            if 1 <= x <= 2:
                return 'Regular'
            elif 3 <= x <= 4:
                return 'Silver'
            elif 5 <= x <= 6:
                return 'Gold'
            else:
                return 'Platinum'
        
        loyalty_level = X_["Transaction_Counts"]
        loyalty_level = loyalty_level.apply(_loyalty_classes_)
        X_["Loyalty_Level"] = loyalty_level
        
        customers_ = X_.Customer_Name.values.tolist()
        levels_ = X_.Loyalty_Level.values.tolist()
        
        function_ = lambda x: [v for k, v in zip(customers_, levels_) if k == x][0]
        
        X["Loyalty_Level"] = X["Customer_Name"].apply(function_)
        X = X.drop(["Customer_Name"], axis=1)
        
        log10_transformed_value = round(np.log10(X["Total_Items"] + 1), 2)
        
        X["Log10_Transformed_Total_Items"] = log10_transformed_value
        X = X.drop(["Total_Items"], axis=1)

        X_categorical_data = X.select_dtypes(include=object)

        if role == "train":
            self.__ohe = skl.preprocessing.OneHotEncoder(sparse_output=False)
            X_ohe = self.__ohe.fit_transform(X_categorical_data)
        else:
            X_ohe = self.__ohe.transform(X_categorical_data)
        
        ohe_features = self.__ohe.get_feature_names_out()
        
        first_cat = "_".join(ohe_features[0].split("_")[:2])
        start_idx = X.columns.tolist().index(first_cat)
        
        X_ohe_df = pd.DataFrame(X_ohe, columns=ohe_features)
        
        for k in range(start_idx, len(ohe_features) + start_idx):
            single_features = np.int16(X_ohe_df[ohe_features[k-start_idx]])
            X.insert(start_idx, ohe_features[k-start_idx], single_features)
        
        X_int = X.select_dtypes(include=[np.int16, np.int32, np.int64])

        if role == "train":
            self.__pca_1 = skl.decomposition.PCA(n_components=1)
            pca_numerical = self.__pca_1.fit_transform(X_int)
        else:
            pca_numerical = self.__pca_1.transform(X_int)
            
        X_pca = pd.DataFrame(pca_numerical, columns=["Feature_1"])
        
        X_log10 = X.Log10_Transformed_Total_Items.reset_index(drop=True)
        
        X_concat = (X_pca, X_log10)
        
        X_dim_reduction_ = pd.concat(X_concat, axis=1)
        
        X_dim_reduction_.columns = ["Feature_1", "Feature_2"]

        X = X_dim_reduction_

        return X, y
        
    def __init__(self, csv_path, target):
        
        # variabel self.__csv_path khusus filepath berekstensi CSV
        self.__csv_path = csv_path
        
        # variabel self.__dataframe khusus DataFrame asal filepath CSV
        dataframe = pd.read_csv(self.__csv_path)

        # variabel fillna_promotion khusus imputasi data missing value
        fillna_promotion = dataframe.Promotion.fillna("Unknown Promotion")

        # variabel dataframe.Promotion khusus penyimpanan variabel pasca imputasi
        # data
        dataframe.Promotion = fillna_promotion

        # variabel dataframe.Transaction_ID diubah menjadi variabel tipe data
        # str
        dataframe.Transaction_ID = dataframe.Transaction_ID.astype(str)

        # variabel dataframe["Datetime"] untuk menyimpan hasil rekayasa fitur
        # Data dan time_of_day
        dataframe["Datetime"] = dataframe.Date + " " + dataframe.time_of_day

        # variabel dataframe diisi dengan dataframe yang sudah diurutkan khusus
        # kolom Datetime dengan syarat diurutkan dari transaksi awal hingga
        # transaksi akhir
        dataframe = dataframe.sort_values("Datetime", ascending=True)

        # variabel dataframe diperbaharui dengan dataframe yang sudah diatur in-
        # deksnya dengan syarat hapus kolom 'index' terlebih dahulu
        dataframe = dataframe.reset_index(drop=True)

        # variabel dataframe diperbaharui dengan penghapusan kolom Transaction_ID
        dataframe = dataframe.drop(columns='Transaction_ID')

        # variabel datetime_series diisi dengan nilai kolom Datetime
        datetime_series = dataframe.Datetime

        # variabel dataframe diperbaharui dengan penghapus dua kolom `Date` dan
        # 'Datetime' dengan syarat penghapusan dua kolom berfokus pada kolom data 
        dataframe = dataframe.drop(columns=["Date", "Datetime"], axis=1)

        # variabel dataframe disisipkan dengan kolom baru `Transaction_Datetime`
        # pada kolom ke-0 dan variabel eksisting `datetime_series`
        dataframe.insert(0, "Transaction_Datetime", datetime_series)

        # variabel timestamp_ khusus penyimpanan struktur data berbasis list
        # dengan prasyarat ekspansif struktur data
        timestamp_ = dataframe.time_of_day.str.split(":", expand=True)

        # variabel timestamp_.columns khusus perubahan kolom dari angka ke 
        # kata secara spesifik
        timestamp_.columns = ['hour', 'minute', 'second']

        # variabel timestamp_.hour khusus perubahan tipe data huruf ke angka 
        # bilangan bulat
        timestamp_.hour = timestamp_.hour.astype(int)

        # fungsi _get_time_classes_ khusus perubahan angka menjadi tipe waktu
        # spesifik
        def _get_time_classes_(x):
            if 0 <= x <= 3:
                return 'Early Morning'
            elif 4 <= x <= 9:
                return 'Morning'
            elif 10 <= x <= 15:
                return 'Midday'
            elif 16 <= x <= 18:
                return 'Afternoon'
            elif 19 <= x <= 21:
                return 'Evening'
            else:
                return 'Late Night'

        # variabel dataframe['TimeClass'] khusus penambahan fitur 
        # TimeClass dengan aplikasi fungsi _get_time_classes_ khusus fitur
        # hour
        dataframe['TimeClass'] = timestamp_.hour.apply(_get_time_classes_)

        # variabel dataframe khusus penghapusan fitur `time_of_day`
        dataframe = dataframe.drop(columns='time_of_day')

        self.__preprocessing_dataframe = dataframe

        self.dataframe = self.get_dataframe(
            split=True, target=target, dropped_columns=target)

        customer_carts = dataframe.Product.str.replace("[", "").str
        customer_carts = customer_carts.replace("]", "").str.split(", ")

        mlb = skl.preprocessing.MultiLabelBinarizer()
        
        customer_carts_mlb = mlb.fit_transform(customer_carts)
        products = ["Buy_" + class_.replace("'", "") for class_ in mlb.classes_]
        
        customer_carts_df = pd.DataFrame(customer_carts_mlb, columns=products)
        
        dataframe_ = dataframe.drop(columns="Product")
        
        for k in range(1, len(products) + 1):
            bool_ = customer_carts_df[products[k-1]]
            dataframe_.insert(k-1, products[k-1], bool_)

        self.__preprocessing_dataframe = dataframe_

        dropped_columns = ["Transaction_Datetime", "City"]
        dropped_columns.append(target)

        split = True
        Xy = self.get_dataframe(
            split=True, target=target, dropped_columns=dropped_columns)

        X_train, X_test, y_train, y_test = Xy

        X_train, y_train = self.__preprocess(X_train, y_train, role="train")
        X_test, y_test = self.__preprocess(X_test, y_test, role="test")

        self.preprocessed_dataframe = (X_train, X_test, y_train, y_test)

## Implementasi Class PreprocessingPipeline

In [3]:
target = "Total_Cost"
dataset = PreprocessingPipeline(filepath, target)

X_train, X_test, y_train, y_test = dataset.preprocessed_dataframe

## Inisiasi _5-Fold Cross Validation_

In [4]:
five_fold_cv = skl.model_selection.KFold(n_splits=5)

## Pencarian Model Terbaik dan Model Tercepat dengan Inisiasi Banyak Model (seperti _Decision Tree Regression_ dkk.)

In [5]:
regression_models = {}
regression_models["Decision Tree"] = skl.tree.DecisionTreeRegressor()
regression_models["Random Forest"] = skl.ensemble.RandomForestRegressor()
regression_models["Gradient Boosting"] = skl.ensemble.GradientBoostingRegressor()
regression_models["K-Nearest Neighbors"] = skl.neighbors.KNeighborsRegressor()
regression_models["Linear Regression"] = skl.linear_model.LinearRegression()
regression_models["Multi Layer Perceptron"] = skl.neural_network.MLPRegressor()
regression_models["Support Vector Machine"] = skl.svm.SVR()

best_overall_model = {}
fastest_overall_model = {}

current_mae_ = np.inf
current_time_ = np.inf

for name, model in regression_models.items():
    print(f"Nama Model :: {name}")

    _ = enumerate(five_fold_cv.split(X_train, y_train))

    mae_folds_ = []

    print("Model sedang pergi ke sekolah untuk belajar selama", end=" ")

    start_time = dttm.now()

    for i, (train_index, test_index) in _:
        # print(f"Fold {i+1}")
        X_train_fold = X_train.iloc[train_index]
        X_test_fold = X_train.iloc[test_index]

        X_train_fold = X_train_fold.select_dtypes(exclude=object)
        X_test_fold = X_test_fold.select_dtypes(exclude=object)
        
        y_train_fold = y_train.iloc[train_index]
        y_test_fold = y_train.iloc[test_index]

        model.fit(X_train_fold, y_train_fold)

        y_predict_fold = model.predict(X_test_fold)
        mae_ = skl.metrics.mean_absolute_error(y_test_fold, y_predict_fold)
        
        mae_folds_.append(mae_)

    end_time = dttm.now()

    diffs = (end_time - start_time).total_seconds()

    print(f"{diffs:.2f} detik dengan rerata selisih mutlak rerata", end=" ")
    print(f"final sebesar {np.mean(mae_folds_):.2f}")

    best_overall_model[name] = [np.mean(mae_folds_), model]
    fastest_overall_model[name] = [diffs, model]

    if np.mean(mae_folds_) < current_mae_:
        current_mae_ = np.mean(mae_folds_)

    if diffs < current_time_:
        current_time_ = diffs

best_function = None
fastest_function = None

for k, v in best_overall_model.items():
    if v[0] == current_mae_:
        print(f"Model yang paling bagus prediksinya :: {k}")
        best_function = v[1]
        break

for k, v in fastest_overall_model.items():
    if v[0] == current_time_:
        print(f"Model yang paling cepat prediksinya :: {k}")
        fastest_function = v[1]
        break

joblib.dump(best_function, "/kaggle/working/best_predictor.pkl")
joblib.dump(fastest_function, "/kaggle/working/quick_predictor.pkl")

Nama Model :: Decision Tree
Model sedang pergi ke sekolah untuk belajar selama 0.25 detik dengan rerata selisih mutlak rerata final sebesar 0.12
Nama Model :: Random Forest
Model sedang pergi ke sekolah untuk belajar selama 13.83 detik dengan rerata selisih mutlak rerata final sebesar 0.11
Nama Model :: Gradient Boosting
Model sedang pergi ke sekolah untuk belajar selama 4.19 detik dengan rerata selisih mutlak rerata final sebesar 0.09
Nama Model :: K-Nearest Neighbors
Model sedang pergi ke sekolah untuk belajar selama 0.11 detik dengan rerata selisih mutlak rerata final sebesar 0.10
Nama Model :: Linear Regression
Model sedang pergi ke sekolah untuk belajar selama 0.04 detik dengan rerata selisih mutlak rerata final sebesar 0.09
Nama Model :: Multi Layer Perceptron
Model sedang pergi ke sekolah untuk belajar selama 2.51 detik dengan rerata selisih mutlak rerata final sebesar 0.09
Nama Model :: Support Vector Machine
Model sedang pergi ke sekolah untuk belajar selama 11.59 detik dengan

['/kaggle/working/quick_predictor.pkl']

## Loading dan Uji Prediksi Model Tercepat dan Model Terbaik 

In [6]:
best_model = joblib.load("/kaggle/working/best_predictor.pkl")
quick_model = joblib.load("/kaggle/working/quick_predictor.pkl")

best_prediction = np.round(best_model.predict(X_test), 2)
quick_prediction = np.round(quick_model.predict(X_test), 2)

_, X_test, _, _ = dataset.dataframe

X_test["Total_Cost"] = np.round(10 * (np.pow(10, best_prediction) - 1), 2)

X_test.to_csv("/kaggle/working/best_prediction.csv", index=False)

## Hasil Prediksi dengan Support Vector Machine (Model Terbaik)

In [7]:
X_test.head()

,Transaction_Datetime,Customer_Name,Product,Total_Items,Payment_Method,City,Customer_Category,Season,Promotion,Member,TimeClass,Total_Cost
14824,2023-03-16 13:46:57,Carol Meyers,"['Jam', 'Dish Soap', 'Bath Salts', 'Energy Dri...",9,Cash,Lisaville,Adult (30-49),Winter,Unknown Promotion,Yes,Midday,85.50
14825,2023-03-16 14:07:49,Susan Kirk,"['Colored Pencils', 'Charcoal', 'Baking Soda',...",7,Debit Card,Beltranshire,Young Adult (20-29),Fall,Discount on Selected Items,No,Midday,65.86
14826,2023-03-16 14:48:11,Steven Cummings,"['Band-Aids', 'Bobby Pins', 'Salt', 'Soda', 'S...",7,Mobile Payment,Port Sarahfurt,Adult (30-49),Spring,BOGO (Buy One Get One),No,Midday,65.86
14827,2023-03-16 15:06:47,Debbie Norton,['Dryer Sheets'],1,Mobile Payment,Wellston,Young Adult (20-29),Winter,Unknown Promotion,Yes,Midday,3.49
14828,2023-03-16 16:18:14,Ricky Baker,"['Dried Cranberries', 'Butter', 'Eggs', 'Sanda...",5,Mobile Payment,Michelleview,Senior (65+),Summer,Unknown Promotion,Yes,Afternoon,47.54


## Hasil Prediksi dengan Linear Regression (Model Tercepat)

In [8]:
X_test["Total_Cost"] = np.round(10 * (np.pow(10, quick_prediction) - 1), 2) 

X_test.to_csv("/kaggle/working/quick_prediction.csv", index=False)

X_test.head()

,Transaction_Datetime,Customer_Name,Product,Total_Items,Payment_Method,City,Customer_Category,Season,Promotion,Member,TimeClass,Total_Cost
14824,2023-03-16 13:46:57,Carol Meyers,"['Jam', 'Dish Soap', 'Bath Salts', 'Energy Dri...",9,Cash,Lisaville,Adult (30-49),Winter,Unknown Promotion,Yes,Midday,90.00
14825,2023-03-16 14:07:49,Susan Kirk,"['Colored Pencils', 'Charcoal', 'Baking Soda',...",7,Debit Card,Beltranshire,Young Adult (20-29),Fall,Discount on Selected Items,No,Midday,65.86
14826,2023-03-16 14:48:11,Steven Cummings,"['Band-Aids', 'Bobby Pins', 'Salt', 'Soda', 'S...",7,Mobile Payment,Port Sarahfurt,Adult (30-49),Spring,BOGO (Buy One Get One),No,Midday,65.86
14827,2023-03-16 15:06:47,Debbie Norton,['Dryer Sheets'],1,Mobile Payment,Wellston,Young Adult (20-29),Winter,Unknown Promotion,Yes,Midday,5.14
14828,2023-03-16 16:18:14,Ricky Baker,"['Dried Cranberries', 'Butter', 'Eggs', 'Sanda...",5,Mobile Payment,Michelleview,Senior (65+),Summer,Unknown Promotion,Yes,Afternoon,44.95


## Pekerjaan Yang Akan Saya Lakukan

* Otomasi Pipeline Pra Pemrosesan Data: Imputasi Data -> _Pre-Splitting Feature Engineering_ -> _Data Splitting_ -> _Post-Splitting Feature Engineering_ -> Visualisasi Data -> _Categorical Data Encoding_ -> _Data Dimensionality Reduction_ -> _Target Engineering_ **[Judul Baru :: Don't Repeat Yourself For Your Pipeline]** **[Sudah Dikerjakan]**
* Permutation Importance Masing-masing Model: _Gradient Boosting_, _Linear Regression_, _Multi-Layer Perceptron_, dan _Support Vector Machine_
* Parallelized Ensemble Regression Antara Model _Gradient Boosting_, _Linear Regression_, _Multi-Layer Perceptron_, dan _Support Vector Machine_
* Sequential Relay Ensemble Regression: _Linear Regression_ -> _Gradient Boosting_ -> _Multi-Layer Perceptron_ -> _Support Vector Machine_
* Dokumentasi Penggunaan Model _Linear Regression_, _Support Vector Machine_, _Stacked Ensemble Regression_, dan _Sequential Relay Ensemble Regression_